# Дедупликация поисковых запросов и технический профиль через API

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DimaKorshikov/rulemma/blob/main/examples/notebooks/02_query_dedup_and_technical_keys.ipynb)

Как схлопнуть неявные дубли поисковых запросов (`зимние кроссовки` = `кроссовки на зиму`) через `POST /v1/lemmatize` и при этом не склеить разные по смыслу товары (`2,6 л.с.` vs `6,2 л.с.`, `100×100 см` vs `100 см`, `без сахара` vs `с сахаром`, `б/у`, `пластиковые окна` vs `пластик для окон`).

### Регистрация и доступ к HTTP API

- Личный кабинет и бесплатный API-ключ: **[rulemma.ru/account/](https://rulemma.ru/account/)** (вход по email, Яндекс ID или Google)
- Документация HTTP API: [rulemma.ru/api/](https://rulemma.ru/api/) · Тарифы и лимиты: [rulemma.ru/pricing/](https://rulemma.ru/pricing/) · OpenAPI: [openapi.yaml](https://rulemma.ru/openapi.yaml)
- Базовый адрес API: `https://rulemma.ru/v1` · Авторизация: заголовок `Authorization: Bearer rlm_live_...` (передавать ключ в URL запрещено)

Как подключить ключ:
1. Создайте бесплатный аккаунт на [rulemma.ru/account/](https://rulemma.ru/account/) и скопируйте токен `rlm_live_...`.
2. Сохраните его в переменную окружения `RULEMMA_API_KEY` (в Google Colab удобно положить токен в панель `Secrets` под именем `RULEMMA_API_KEY` или ввести в ячейке настройки).
3. Без ключа базовые примеры работают через `/v1/sandbox/test` (до 10 строк); расширенные опции (`compact_codes`, `keep_words`, `overrides`) требуют бесплатный токен с [rulemma.ru/account/](https://rulemma.ru/account/).

### Содержание тетрадки

1. Подключение к API `rulemma.ru`
2. Группировка явных и неявных дублей по `semantic_key`
3. Защита характеристик в `profile="technical"` (дроби, габариты, слэши, вольты)
4. Связки предлогов, защита главного слова и пользовательские `overrides`
5. Итоговая таблица кластеров (готова к экспорту в CSV/Excel)

## 1. Клиент HTTP API

In [ ]:
import json
import os
import urllib.error
import urllib.request

BASE_URL = os.environ.get("RULEMMA_API_BASE", "https://rulemma.ru/v1").rstrip("/")
API_KEY = os.environ.get("RULEMMA_API_KEY", "").strip()

# В Google Colab ключ можно автоматически прочитать из вкладки Secrets
if not API_KEY:
    try:
        from google.colab import userdata  # type: ignore

        API_KEY = (userdata.get("RULEMMA_API_KEY") or "").strip()
    except Exception:
        pass

HAS_KEY = bool(API_KEY)
print("Базовый URL:", BASE_URL)
print(
    "Режим:",
    "боевой API с Bearer-токеном" if HAS_KEY else "публичная песочница без ключа (до 10 строк)",
)
if not HAS_KEY:
    print("Получить бесплатный API-ключ: https://rulemma.ru/account/")


def api_request(
    method: str,
    path: str,
    payload: dict | None = None,
    *,
    auth: bool = True,
    timeout: int = 20,
) -> tuple[dict, dict[str, str]]:
    """Выполняет HTTP-запрос к rulemma API без сторонних зависимостей."""
    url = f"{BASE_URL}{path}"
    headers = {
        "Accept": "application/json",
        "User-Agent": "rulemma-jupyter-example/1.0",
    }
    if auth:
        if not API_KEY:
            raise RuntimeError(
                "Требуется RULEMMA_API_KEY. Выпустите бесплатный токен на https://rulemma.ru/account/"
            )
        headers["Authorization"] = f"Bearer {API_KEY}"

    body = None
    if payload is not None:
        body = json.dumps(payload, ensure_ascii=False).encode("utf-8")
        headers["Content-Type"] = "application/json; charset=utf-8"

    import time as _time

    req = urllib.request.Request(url, data=body, headers=headers, method=method)
    for attempt in range(2):
        try:
            with urllib.request.urlopen(req, timeout=timeout) as resp:
                resp_headers = {k.lower(): v for k, v in resp.headers.items()}
                return json.loads(resp.read().decode("utf-8")), resp_headers
        except urllib.error.HTTPError as exc:
            raw = exc.read().decode("utf-8", errors="replace")
            if exc.code == 429 and attempt == 0:
                wait = min(12, max(1, int(exc.headers.get("Retry-After", "4"))))
                print(f"Лимит скорости (HTTP 429), пауза {wait} с...")
                _time.sleep(wait)
                continue
            raise RuntimeError(f"HTTP {exc.code} {path}: {raw}") from exc
    raise RuntimeError(f"Не удалось выполнить {method} {path}")

## 2. Дедупликация поискового ядра с `profile="technical"`

Обычный стеммер или лемматизатор разносит «зимние кроссовки» и «кроссовки на зиму» в разные группы. Поле `semantic_key` сводит относительные прилагательные к существительным и сортирует значимые токены.

In [ ]:
from collections import defaultdict

queries = [
    "зимние кроссовки найк",
    "кроссовки на зиму найк",
    "купить кроссовки найк зимние",
    "детский рюкзак для школы",
    "рюкзак для детей школьный",
    "кофемашина делонги зерновая",
    "зерновые кофемашины делонги",
    "зимние ботинки б/у",
    "зимние ботинки",
]

payload = {
    "queries": queries,
    "profile": "technical",
    "normalize_homoglyphs": True,
}

if HAS_KEY:
    resp, hdrs = api_request("POST", "/lemmatize", payload)
    print(f"Списано единиц: {resp['units_charged']} (остаток: {hdrs.get('x-quota-remaining', '?')})")
else:
    resp, _ = api_request("POST", "/sandbox/test", {"method": "lemmatize", **payload}, auth=False)
    print("Работаем через публичную песочницу (без ключа, до 10 строк).")

clusters = defaultdict(list)
for item in resp["items"]:
    clusters[item["semantic_key"]].append(item["input"])

print(f"\nИз {len(queries)} запросов получено {len(clusters)} уникальных кластеров:")
for key, group in clusters.items():
    print(f"  [{key}] ({len(group)} шт.)")
    for q in group:
        print(f"     - {q}")

## 3. Проверка критичных коммерческих различий

Проверим пары, которые наивная токенизация ошибочно склеивает, а `profile="technical"` держит строго раздельно (и наоборот — эквиваленты вроде `0,5 л` и `0.5 л` склеивает).

In [ ]:
check_queries = [
    "бензопила 2,6 л.с.",
    "бензопила 6,2 л.с.",
    "плитка 100×100 см",
    "плитка 100 см",
    "термос 0,5 л",
    "термос 0.5 л",
    "винт 5 мм",
    "винт 5мм",
    "шоколад без сахара",
    "шоколад с сахаром",
]

if HAS_KEY:
    r, _ = api_request("POST", "/lemmatize", {"queries": check_queries, "profile": "technical"})
else:
    r, _ = api_request(
        "POST",
        "/sandbox/test",
        {"method": "lemmatize", "queries": check_queries, "profile": "technical"},
        auth=False,
    )

by_input = {row["input"]: row["semantic_key"] for row in r["items"]}
for a, b, want_equal in [
    ("бензопила 2,6 л.с.", "бензопила 6,2 л.с.", False),
    ("плитка 100×100 см", "плитка 100 см", False),
    ("шоколад без сахара", "шоколад с сахаром", False),
    ("термос 0,5 л", "термос 0.5 л", True),
    ("винт 5 мм", "винт 5мм", True),
]:
    eq = by_input[a] == by_input[b]
    status = "OK" if eq == want_equal else "DIFF"
    rel = "==" if eq else "!="
    print(f"[{status}] {a!r} {rel} {b!r}  ->  ({by_input[a]!r} vs {by_input[b]!r})")

## 4. Связки предлогов, защита главного слова и доменные `overrides`

Флаги `preserve_prep_bindings`, `bind_purpose_prep`, `keep_words`, `compact_codes` и словарь `overrides` (до 500 замен прямо в запросе) убирают тонкие коллизии каталога.

In [ ]:
if HAS_KEY:
    adv_resp, _ = api_request(
        "POST",
        "/lemmatize",
        {
            "queries": [
                "кофе без сахара с молоком",
                "кофе с сахаром без молока",
                "пластиковые окна",
                "пластик для окон",
                "зеркальный пластик",
                "пластиковое зеркало",
                "кроссовки nike p 6000",
                "кроссовки nike p-6000",
                "ушм бош 125 мм",
            ],
            "profile": "technical",
            "preserve_prep_bindings": True,
            "bind_purpose_prep": True,
            "compact_codes": True,
            "keep_words": ["пластик"],
            "overrides": {"ушм": "болгарка"},
        },
    )
    for row in adv_resp["items"]:
        print(f"{row['input']:28s} | lem={row['lemmatized']:24s} | sem={row['semantic_key']}")
else:
    # В песочнице доступны preserve_prep_bindings и bind_purpose_prep
    sbx, _ = api_request(
        "POST",
        "/sandbox/test",
        {
            "method": "lemmatize",
            "queries": [
                "кофе без сахара с молоком",
                "кофе с сахаром без молока",
                "пластиковые окна",
                "пластик для окон",
            ],
            "profile": "technical",
            "preserve_prep_bindings": True,
            "bind_purpose_prep": True,
        },
        auth=False,
    )
    for row in sbx["items"]:
        print(f"{row['input']:28s} | sem={row['semantic_key']}")
    print("\nДля compact_codes, keep_words и overrides получите бесплатный ключ: https://rulemma.ru/account/")

---

## Что дальше

- Следующая тетрадка: [03_morphology_inflection_and_seo_titles.ipynb](03_morphology_inflection_and_seo_titles.ipynb)
- Все Jupyter-тетрадки с HTTP API: [examples/notebooks/README.md](README.md)
- Регистрация и управление API-ключами: [rulemma.ru/account/](https://rulemma.ru/account/)
- Документация эндпоинтов и форматов ошибок: [rulemma.ru/api/](https://rulemma.ru/api/) · [openapi.yaml](https://rulemma.ru/openapi.yaml)
- Нужны миллионы строк локально без сети (Go / Python CGO / CLI)? Смотрите локальные примеры [examples/01..13](../README.md).